In [3]:
import apertium
from apertium import Analyzer
from typing import List
import subprocess
analyzer = Analyzer('mar')  # загрузить словарь маратхи один раз


In [4]:
def _lu_has_tag(lu, tag: str) -> bool:
    """
    True if any subreading contains the given morphological tag.
    """
    readings = getattr(lu, "readings", None) or []
    for reading in readings:          # reading: List[SReading]
        for sub in reading:           # sub: SReading(baseform, tags)
            if tag in (getattr(sub, "tags", None) or []):
                return True
    return False


def _lu_is_unknown(lu) -> bool:
    """
    Apertium stream format uses special symbols to mark unknown/unavailable analyses.
    In streamparser these are exposed via lu.knownness.symbol (e.g. '*', '@', '#').
    """
    knownness = getattr(lu, "knownness", None)
    symbol = getattr(knownness, "symbol", "")
    return symbol in ("*", "@", "#")  # unknown / biunknown / genunknown


def _lu_lemma_candidates(lu) -> List[str]:
    """
    Extract lemma candidates from lu.readings without parsing string representations.
    We take the baseform of the first subreading of each reading.
    """
    readings = getattr(lu, "readings", None) or []
    cands: List[str] = []

    for reading in readings:
        if not reading:
            continue
        base = getattr(reading[0], "baseform", "")
        if not base:
            continue
        # Just in case: strip leading unknown markers if they appear in baseform.
        if base[0] in ("*", "@", "#"):
            base = base[1:]
        cands.append(base)

    # Deduplicate while preserving order.
    seen = set()
    uniq: List[str] = []
    for x in cands:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq


def _pick_matching_lu(lus, token: str):
    """
    Prefer a LexicalUnit whose wordform exactly matches the token; otherwise fallback to first.
    """
    for lu in lus:
        if getattr(lu, "wordform", None) == token:
            return lu
    return lus[0] if lus else None


def lemmatize_tokens(tokens: List[str]) -> List[str]:
    """
    Lemmatize a list of tokens using Apertium Analyzer.

    Strategy:
    - Analyze the whole token sequence in one call for speed.
    - Align output LexicalUnits with input tokens.
    - If alignment breaks, fallback to per-token analysis for that token.
    """
    if not tokens:
        return []

    text = " ".join(tokens)

    try:
        lus = analyzer.analyze(text)  # List[LexicalUnit]
    except getattr(apertium, "ModeNotInstalled", Exception) as e:
        raise RuntimeError(
            "Apertium mode for this language is not installed (ModeNotInstalled). "
            "Install the Marathi module and ensure Apertium can find it."
        ) from e
    except (FileNotFoundError, OSError) as e:
        raise RuntimeError(
            "Failed to run Apertium executable. "
            "Make sure Apertium is installed and available in PATH (or via the apertium installer)."
        ) from e
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Apertium process failed with exit code {e.returncode}."
        ) from e
    except Exception as e:
        raise RuntimeError("Unexpected error while running Apertium analyzer.") from e

    out: List[str] = []
    i = 0  # index in lus

    for tok in tokens:
        # Skip injected sentence boundary markers like "./.<sent>" when they don't correspond to a real token.
        while i < len(lus):
            lu = lus[i]
            if getattr(lu, "wordform", None) == "." and _lu_has_tag(lu, "sent") and tok != ".":
                i += 1
                continue
            break

        if i >= len(lus):
            out.append(tok)
            continue

        lu = lus[i]

        # If alignment is off (tokenization differs), fallback to per-token analyze.
        if getattr(lu, "wordform", None) != tok:
            single = analyzer.analyze(tok)
            lu2 = _pick_matching_lu(single, tok)
            if lu2 is None or _lu_is_unknown(lu2):
                out.append(tok)
            else:
                cands = _lu_lemma_candidates(lu2)
                out.append(cands[0] if cands else tok)
            continue

        # Normal aligned case.
        i += 1
        if _lu_is_unknown(lu):
            out.append(tok)
            continue

        cands = _lu_lemma_candidates(lu)
        out.append(cands[0] if cands else tok)

    return out


In [7]:
test_tokens = ["मुलगा", "मुलगे", "मुलाला", "मुलाने", "मुलांचा"]
lemmatize_tokens(test_tokens)

['मुलगा', 'मुलगे', 'मुल', 'मुल', 'मुल']